# Sprint 1 — Construcción de Master Table

Este notebook construye una **Master Table analítica** para el proyecto:

**Identificación de Clientes Premium — Olist**

El objetivo es dejar una tabla base lista para el EDA, con variables derivadas tipo RFM:

- `total_spent`
- `total_orders`
- `avg_ticket`
- `recency_days`
- `avg_review_score`
- `is_premium`

La salida principal será:

```text
data/processed/master_table.parquet
```


## 1. Importar librerías

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 2. Definir rutas del proyecto

Este notebook asume que se ejecuta desde la carpeta `notebooks/sprint_01/`.

Si lo ejecutas desde otra ubicación, ajusta `PROJECT_ROOT`.


In [12]:
PROJECT_ROOT = Path("../..").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PARQUET_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw dir:", RAW_DIR)
print("Parquet dir:", PARQUET_DIR)
print("Processed dir:", PROCESSED_DIR)


Project root: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum
Raw dir: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/raw
Parquet dir: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/raw
Processed dir: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/processed


## 3. Cargar datasets

Se prioriza cargar desde `data/parquet/`.  
Si no existen los archivos Parquet, se cargan desde `data/raw/`.


In [13]:
def load_dataset(file_stem: str) -> pd.DataFrame:
    """Load a dataset from Parquet if available; otherwise from CSV."""
    parquet_path = PARQUET_DIR / f"{file_stem}.parquet"
    csv_path = RAW_DIR / f"{file_stem}.csv"

    if parquet_path.exists():
        print(f"Loading Parquet: {parquet_path.name}")
        return pd.read_parquet(parquet_path)

    if csv_path.exists():
        print(f"Loading CSV: {csv_path.name}")
        return pd.read_csv(csv_path)

    raise FileNotFoundError(f"Dataset not found: {file_stem}")


customers = load_dataset("olist_customers_dataset")
orders = load_dataset("olist_orders_dataset")
payments = load_dataset("olist_order_payments_dataset")
reviews = load_dataset("olist_order_reviews_dataset")
order_items = load_dataset("olist_order_items_dataset")
products = load_dataset("olist_products_dataset")
sellers = load_dataset("olist_sellers_dataset")


Loading Parquet: olist_customers_dataset.parquet
Loading Parquet: olist_orders_dataset.parquet
Loading Parquet: olist_order_payments_dataset.parquet
Loading Parquet: olist_order_reviews_dataset.parquet
Loading Parquet: olist_order_items_dataset.parquet
Loading Parquet: olist_products_dataset.parquet
Loading Parquet: olist_sellers_dataset.parquet


## 4. Validación inicial de datasets

In [14]:
datasets = {
    "customers": customers,
    "orders": orders,
    "payments": payments,
    "reviews": reviews,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
}

summary = []

for name, df in datasets.items():
    summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum(),
        "total_nulls": df.isna().sum().sum(),
    })

pd.DataFrame(summary)


,dataset,rows,columns,duplicated_rows,total_nulls
0,customers,99441,5,0,0
1,orders,99441,8,0,4908
2,payments,103886,5,0,0
3,reviews,99224,7,0,145903
4,order_items,112650,7,0,0
5,products,32951,9,0,2448
6,sellers,3095,4,0,0


## 5. Convertir fechas

Las fechas son necesarias para calcular `recency_days` y variables de entrega.


In [15]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].head()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


## 6. Construir variables de pago por pedido

`payment_value` representa el valor pagado por orden.  
Una orden puede tener más de un registro de pago, por eso se agrupa por `order_id`.


In [16]:
order_payments = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        order_payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_methods_count=("payment_type", "nunique"),
        main_payment_type=("payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    )
)

order_payments.head()


,order_id,order_payment_value,payment_installments,payment_methods_count,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1,credit_card


## 7. Construir variables de items por pedido

Estas variables permiten complementar el gasto con precio, flete y número de productos.


In [17]:
order_item_features = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_items_count=("order_item_id", "count"),
        order_products_count=("product_id", "nunique"),
        order_price_total=("price", "sum"),
        order_freight_total=("freight_value", "sum"),
        sellers_count=("seller_id", "nunique"),
    )
)

order_item_features["freight_ratio"] = (
    order_item_features["order_freight_total"] /
    (order_item_features["order_price_total"] + order_item_features["order_freight_total"])
).replace([np.inf, -np.inf], np.nan)

order_item_features.head()


,order_id,order_items_count,order_products_count,order_price_total,order_freight_total,sellers_count,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,58.90,13.29,1,0.184098
1,00018f77f2f0320c557190d7a144bdd3,1,1,239.90,19.93,1,0.076704
2,000229ec398224ef6ca0657da4fc703e,1,1,199.00,17.87,1,0.082400
3,00024acbcdf0a6daa1e931b038114c75,1,1,12.99,12.79,1,0.496121
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,199.90,18.14,1,0.083196


## 8. Construir variables de reviews por pedido

In [18]:
order_reviews = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "nunique"),
    )
)

order_reviews.head()


,order_id,review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


## 9. Crear tabla de órdenes enriquecida

Aquí se une `orders` con pagos, items y reviews a nivel de orden.


In [19]:
orders_enriched = (
    orders
    .merge(order_payments, on="order_id", how="left")
    .merge(order_item_features, on="order_id", how="left")
    .merge(order_reviews, on="order_id", how="left")
)

orders_enriched["delivery_days"] = (
    orders_enriched["order_delivered_customer_date"] -
    orders_enriched["order_purchase_timestamp"]
).dt.days

orders_enriched["estimated_delivery_days"] = (
    orders_enriched["order_estimated_delivery_date"] -
    orders_enriched["order_purchase_timestamp"]
).dt.days

orders_enriched["is_delivered"] = np.where(
    orders_enriched["order_status"].eq("delivered"),
    1,
    0,
)

orders_enriched["is_canceled"] = np.where(
    orders_enriched["order_status"].eq("canceled"),
    1,
    0,
)

orders_enriched["is_late_delivery"] = np.where(
    orders_enriched["order_delivered_customer_date"] > orders_enriched["order_estimated_delivery_date"],
    1,
    0,
)

orders_enriched.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_payment_value,payment_installments,payment_methods_count,main_payment_type,order_items_count,order_products_count,order_price_total,order_freight_total,sellers_count,freight_ratio,review_score,review_count,delivery_days,estimated_delivery_days,is_delivered,is_canceled,is_late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,38.71,1.0,2.0,voucher,1.0,1.0,29.99,8.72,1.0,0.225265,4.0,1.0,8.0,15,1,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,141.46,1.0,1.0,boleto,1.0,1.0,118.70,22.76,1.0,0.160894,4.0,1.0,13.0,19,1,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,179.12,3.0,1.0,credit_card,1.0,1.0,159.90,19.22,1.0,0.107302,5.0,1.0,9.0,26,1,0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,72.20,1.0,1.0,credit_card,1.0,1.0,45.00,27.20,1.0,0.376731,5.0,1.0,13.0,26,1,0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,28.62,1.0,1.0,credit_card,1.0,1.0,19.90,8.72,1.0,0.304682,5.0,1.0,2.0,12,1,0,0


## 10. Construir features por cliente

Se agregan los pedidos a nivel de `customer_id`.


In [20]:
reference_date = orders_enriched["order_purchase_timestamp"].max()

customer_features = (
    orders_enriched
    .groupby("customer_id", as_index=False)
    .agg(
        total_spent=("order_payment_value", "sum"),
        total_orders=("order_id", "nunique"),
        total_items=("order_items_count", "sum"),
        total_products=("order_products_count", "sum"),
        avg_ticket=("order_payment_value", "mean"),
        avg_order_price=("order_price_total", "mean"),
        avg_freight_value=("order_freight_total", "mean"),
        avg_freight_ratio=("freight_ratio", "mean"),
        avg_review_score=("review_score", "mean"),
        total_reviews=("review_count", "sum"),
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_estimated_delivery_days=("estimated_delivery_days", "mean"),
        delivered_orders=("is_delivered", "sum"),
        canceled_orders=("is_canceled", "sum"),
        late_deliveries=("is_late_delivery", "sum"),
        payment_methods_count=("payment_methods_count", "max"),
        main_payment_type=("main_payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    )
)

customer_features["recency_days"] = (
    reference_date - customer_features["last_purchase"]
).dt.days

customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase"] - customer_features["first_purchase"]
).dt.days

customer_features["cancellation_rate"] = (
    customer_features["canceled_orders"] / customer_features["total_orders"]
)

customer_features["late_delivery_rate"] = (
    customer_features["late_deliveries"] / customer_features["total_orders"]
)

customer_features.head()


,customer_id,total_spent,total_orders,total_items,total_products,avg_ticket,avg_order_price,avg_freight_value,avg_freight_ratio,avg_review_score,total_reviews,first_purchase,last_purchase,avg_delivery_days,avg_estimated_delivery_days,delivered_orders,canceled_orders,late_deliveries,payment_methods_count,main_payment_type,recency_days,customer_lifetime_days,cancellation_rate,late_delivery_rate
0,00012a2ce6f8dcda20d059ce98491703,114.74,1,1.0,1.0,114.74,89.80,24.94,0.217361,1.0,1.0,2017-11-14 16:08:26,2017-11-14 16:08:26,13.0,19.0,1,0,0,1.0,credit_card,337,0,0.0,0.0
1,000161a058600d5901f007fab4c27140,67.41,1,1.0,1.0,67.41,54.90,12.51,0.185581,4.0,1.0,2017-07-16 09:40:32,2017-07-16 09:40:32,9.0,18.0,1,0,0,1.0,credit_card,458,0,0.0,0.0
2,0001fd6190edaaf884bcaf3d49edf079,195.42,1,1.0,1.0,195.42,179.99,15.43,0.078958,5.0,1.0,2017-02-28 11:06:43,2017-02-28 11:06:43,5.0,21.0,1,0,0,1.0,credit_card,596,0,0.0,0.0
3,0002414f95344307404f0ace7a26f1d5,179.35,1,1.0,1.0,179.35,149.90,29.45,0.164204,5.0,1.0,2017-08-16 13:09:20,2017-08-16 13:09:20,28.0,28.0,1,0,0,1.0,boleto,427,0,0.0,0.0
4,000379cdec625522490c315e70c7a9fb,107.01,1,1.0,1.0,107.01,93.00,14.01,0.130922,4.0,1.0,2018-04-02 13:42:17,2018-04-02 13:42:17,11.0,15.0,1,0,0,1.0,boleto,198,0,0.0,0.0


## 11. Construir Master Table final

Se une la información transaccional con datos geográficos del cliente.


In [21]:
master_table = (
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state",
        ]
    ]
    .merge(customer_features, on="customer_id", how="left")
)

numeric_cols = master_table.select_dtypes(include=["number"]).columns
master_table[numeric_cols] = master_table[numeric_cols].fillna(0)

master_table.head()


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_spent,total_orders,total_items,total_products,avg_ticket,avg_order_price,avg_freight_value,avg_freight_ratio,avg_review_score,total_reviews,first_purchase,last_purchase,avg_delivery_days,avg_estimated_delivery_days,delivered_orders,canceled_orders,late_deliveries,payment_methods_count,main_payment_type,recency_days,customer_lifetime_days,cancellation_rate,late_delivery_rate
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,146.87,1,1.0,1.0,146.87,124.99,21.88,0.148975,4.0,1.0,2017-05-16 15:05:35,2017-05-16 15:05:35,8.0,19.0,1,0,0,1.0,credit_card,519,0,0.0,0.0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,335.48,1,1.0,1.0,335.48,289.00,46.48,0.138548,5.0,1.0,2018-01-12 20:48:24,2018-01-12 20:48:24,16.0,24.0,1,0,0,1.0,credit_card,277,0,0.0,0.0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,157.73,1,1.0,1.0,157.73,139.94,17.79,0.112788,5.0,1.0,2018-05-19 16:07:45,2018-05-19 16:07:45,26.0,24.0,1,0,1,1.0,credit_card,151,0,0.0,1.0
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,173.30,1,1.0,1.0,173.30,149.94,23.36,0.134795,5.0,1.0,2018-03-13 16:06:38,2018-03-13 16:06:38,14.0,27.0,1,0,0,1.0,credit_card,218,0,0.0,0.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,252.25,1,1.0,1.0,252.25,230.00,22.25,0.088206,5.0,1.0,2018-07-29 09:51:30,2018-07-29 09:51:30,11.0,16.0,1,0,0,1.0,credit_card,80,0,0.0,0.0


## 12. Crear target preliminar `is_premium`

Definición inicial:

> Un cliente premium es aquel cuyo `total_spent` se encuentra en el percentil 80 superior del dataset.

Esta definición es preliminar y puede ajustarse en Sprint 2.


In [22]:
premium_threshold = master_table["total_spent"].quantile(0.80)

master_table["is_premium"] = np.where(
    master_table["total_spent"] >= premium_threshold,
    1,
    0,
)

print("Premium threshold:", premium_threshold)

master_table["is_premium"].value_counts(normalize=True).rename("proportion")


Premium threshold: 202.76


is_premium
0    0.799912
1    0.200088
Name: proportion, dtype: float64

## 13. Validación rápida de la Master Table

In [23]:
print("Rows:", master_table.shape[0])
print("Columns:", master_table.shape[1])
print("Duplicated customer_id:", master_table["customer_id"].duplicated().sum())
print("Null values:", master_table.isna().sum().sum())

master_table.describe(include="all").T.head(30)


Rows: 99441
Columns: 29
Duplicated customer_id: 0
Null values: 1


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_id,99441,99441,06b8999e2fba1a1fbc88172c00ba8bc7,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_unique_id,99441,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,99441.0,NaN,NaN,NaN,35137.474583,1003.0,11347.0,24416.0,58900.0,99990.0,29797.938996
customer_city,99441,4119,sao paulo,15540,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_state,99441,27,SP,41746,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_spent,99441.0,NaN,NaN,NaN,160.988648,0.0,62.01,105.29,176.97,13664.08,221.950728
total_orders,99441.0,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,0.0
total_items,99441.0,NaN,NaN,NaN,1.132833,0.0,1.0,1.0,1.0,21.0,0.545666
total_products,99441.0,NaN,NaN,NaN,1.030008,0.0,1.0,1.0,1.0,8.0,0.243344
avg_ticket,99441.0,NaN,NaN,NaN,160.988648,0.0,62.01,105.29,176.97,13664.08,221.950728


## 14. Métricas preliminares de negocio

In [24]:
business_metrics = master_table.groupby("is_premium").agg(
    customers=("customer_id", "count"),
    avg_total_spent=("total_spent", "mean"),
    median_total_spent=("total_spent", "median"),
    avg_total_orders=("total_orders", "mean"),
    avg_ticket=("avg_ticket", "mean"),
    avg_review_score=("avg_review_score", "mean"),
    avg_recency_days=("recency_days", "mean"),
)

business_metrics


,customers,avg_total_spent,median_total_spent,avg_total_orders,avg_ticket,avg_review_score,avg_recency_days
is_premium,,,,,,,
0,79544,93.712981,85.495,1.0,93.712981,4.088859,289.761415
1,19897,429.942540,308.130,1.0,429.942540,3.920792,290.454993


## 15. Exportar Master Table

Se guarda en Parquet para que el resto del equipo pueda trabajar en paralelo.


In [25]:
output_file = PROCESSED_DIR / "master_table.parquet"

master_table.to_parquet(
    output_file,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

print(f"Master Table saved to: {output_file}")
print(f"File size: {output_file.stat().st_size / (1024 * 1024):.2f} MB")


Master Table saved to: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/processed/master_table.parquet
File size: 10.47 MB


## 16. Columnas generadas

In [26]:
master_table.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'total_spent',
 'total_orders',
 'total_items',
 'total_products',
 'avg_ticket',
 'avg_order_price',
 'avg_freight_value',
 'avg_freight_ratio',
 'avg_review_score',
 'total_reviews',
 'first_purchase',
 'last_purchase',
 'avg_delivery_days',
 'avg_estimated_delivery_days',
 'delivered_orders',
 'canceled_orders',
 'late_deliveries',
 'payment_methods_count',
 'main_payment_type',
 'recency_days',
 'customer_lifetime_days',
 'cancellation_rate',
 'late_delivery_rate',
 'is_premium']

## 17. Próximo paso

Con este archivo ya generado, el equipo puede avanzar con:

```text
data/processed/master_table.parquet
```

La persona encargada del EDA puede cargarlo directamente:

```python
import pandas as pd

master_table = pd.read_parquet("data/processed/master_table.parquet")
```
